In [1]:
import pandas as pd
from tensorflow.python.ops.gen_data_flow_ops import barrier_incomplete_size_eager_fallback

I0000 00:00:1790153341.292453    7569 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790153341.322627    7569 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import pandas as pd
import glob
import os
from huggingface_hub import HfFileSystem

# ==========================================
# 1. LOAD LOCAL METADATA (MEMORY-SAFE MODE)
# ==========================================
print("Loading local Metadata (Bypassing RAM limits)...")
folder_path = "./Amazon_review_ds_elec/"
file_list = glob.glob(os.path.join(folder_path, "*.parquet"))

# ONLY load these 4 columns to prevent OSError 12
needed_columns = ['parent_asin', 'title', 'store', 'main_category']
final_df_list = [pd.read_parquet(file, columns=needed_columns) for file in file_list]
full_final_df = pd.concat(final_df_list, ignore_index=True)
print(f"Loaded {len(full_final_df):,} metadata rows safely.")

# ==========================================
# 2. DYNAMICALLY FIND & STREAM ONE REVIEWS CHUNK
# ==========================================
# print("\nSearching Hugging Face for the exact Review Parquet paths...")
# fs = HfFileSystem()
#
# # The ** wildcard bypasses the folder guessing game entirely
# review_files = fs.glob("datasets/McAuley-Lab/Amazon-Reviews-2023/**/raw_review_Electronics/*.parquet")
# first_file = "hf://" + review_files[0]
# print(f"Found the reviews! Streaming into pandas: {first_file}")

# read_parquet natively streams from hf:// paths without full disk downloads
# reviews_chunk = pd.read_parquet(first_file)

# Take a 100k sample to keep the merge fast
# reviews_sample = reviews_chunk.sample(n=100000, random_state=42)
# print(f"Loaded {len(reviews_sample):,} reviews successfully.")
#
# # ==========================================
# # 3. MERGE THE GRAPH EDGES
# # ==========================================
# merged_final_df = pd.merge(
#     reviews_sample,
#     full_final_df,
#     on='parent_asin',
#     how='inner'
# )
#
# final_final_df = merged_final_df.rename(columns={
#     'store': 'brand',
#     'main_category': 'category',
#     'text': 'reviews_text',
#     'title': 'name'
# })

# print(f"\nFinal dataset ready! Total matched rows: {len(final_final_df):,}")
# display(final_final_df.head(3))

Loading local Metadata (Bypassing RAM limits)...
Loaded 1,610,012 metadata rows safely.


In [3]:

import pandas as pd

# 1. Point this to the 6GB file you just downloaded
reviews_file_path = "./Electronics.jsonl.gz"

print("Loading a 100k sample from the local 6GB reviews file...")

# 2. Use chunksize to read just the first 100,000 rows without exploding your RAM
json_reader = pd.read_json(reviews_file_path, lines=True, chunksize=100000)
reviews_sample = next(json_reader)

print(f"Successfully loaded {len(reviews_sample):,} reviews into memory!")

# 3. Merge with your local metadata (full_final_df) that you already loaded successfully
merged_final_df = pd.merge(
    reviews_sample,
    full_final_df,
    on='parent_asin',
    how='inner'
)

# 4. Rename columns to match your existing Keras pipeline inputs
final_df = merged_final_df.rename(columns={
    'store': 'brand',
    'main_category': 'category',
    'text': 'reviews_text',
    'title': 'name'
})



print(f"\nFinal dataset ready! Total matched rows: {len(final_df):,}")
display(final_df.head(3))

Loading a 100k sample from the local 6GB reviews file...
Successfully loaded 100,000 reviews into memory!

Final dataset ready! Total matched rows: 100,000


,rating,title_x,reviews_text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,title_y,brand,category
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True,"Binoculars, 12x42 Binoculars for Adults, Binoc...",Hikkogo,Camera & Photo
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True,"Toys for 4-5 Year Old Boys, Mom&myaboys 8 X 21...",mom&myaboys,Camera & Photo
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True,"Senso Bluetooth Headphones, Best Wireless Spor...",Senso,All Electronics


In [4]:
# 1. Count how many ratings each item (parent_asin) has received
item_counts = final_df['parent_asin'].value_counts()

# 2. Define the threshold: items with fewer than 5 ratings are "Cold"
cold_items = item_counts[item_counts < 5].index.tolist()
head_items = item_counts[item_counts >= 5].index.tolist()

print(f"Total Unique Items: {len(item_counts):,}")
print(f"Head Items (5+ ratings): {len(head_items):,}")
print(f"Cold Items (<5 ratings): {len(cold_items):,}")

# 3. Create explicit evaluation datasets based on the split
final_df_cold = final_df[final_df['parent_asin'].isin(cold_items)].copy()
final_df_head = final_df[final_df['parent_asin'].isin(head_items)].copy()

print(f"\nYour 'Cold-Start' evaluation set contains {len(final_df_cold):,} rows.")

Total Unique Items: 59,429
Head Items (5+ ratings): 2,434
Cold Items (<5 ratings): 56,995

Your 'Cold-Start' evaluation set contains 74,000 rows.


In [5]:
final_df['user_id_encoded'] = final_df['user_id'].astype('category').cat.codes
final_df['item_id_encoded'] = final_df['parent_asin'].astype('category').cat.codes

# Rename the product title from the metadata to 'name'
final_df = final_df.rename(columns={'title_y': 'name'})

# Verify it worked
# print(final_df[['name', 'reviews_text', 'brand_encoded', 'category_encoded']].head(2))

In [6]:
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
def build_joint_explainability_model(num_entities, num_relations, vocab_size, max_seq_len = 128, embed_dim = 64):
    input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'input_ids')
    attn_mask = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name = 'attention_mask')


    bert = TFBertModel.from_pretrained('bert-base-uncased')
    bert.trainable = False
    bert_output = bert(input_ids = input_ids, attention_mask = attn_mask)
    bert_pooled = bert_output.pooler_output

    bert_compressed = tf.keras.layers.Dense(64, activation='relu', name='bert_compression')(bert_pooled)
    head_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'head_idx')
    # rel_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'rel_idx')
    tail_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'tail_idx')
    brand_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'brand_idx')
    cat_idx = tf.keras.layers.Input(shape=(1,), dtype="int32", name = 'cat_idx')


    entity_emb_layer = tf.keras.layers.Embedding(input_dim = num_entities, output_dim = embed_dim, name = 'entity_emb')
    # relation_emb_layer = tf.keras.layers.Embedding(input_dim = num_relations, output_dim = embed_dim, name = 'relation_emb')



    h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
    # r_emb = tf.keras.layers.Flatten()(relation_emb_layer(rel_idx))
    t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))
    b_emb = tf.keras.layers.Flatten()(entity_emb_layer(brand_idx))
    c_emb = tf.keras.layers.Flatten()(entity_emb_layer(cat_idx))

    merged = tf.keras.layers.concatenate([h_emb, t_emb, b_emb, c_emb, bert_compressed])


    fusion_dense = tf.keras.layers.Dense(256, activation= 'relu', name = 'fusion_dense')(merged)
    fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)


    rating_output = tf.keras.layers.Dense(1, activation= 'linear', name = 'rating_output')(fusion_state)




    decoder_inputs = tf.keras.layers.Input(shape = (None,), dtype="int32", name = 'decoder_inputs')

    lstm_initial_state = tf.keras.layers.Dense(512, activation= 'tanh', name = 'lstm_initial_state')(fusion_state)

    decoder_embedding = tf.keras.layers.Embedding(input_dim = vocab_size, output_dim = 256, name = 'decoder_embedding')(decoder_inputs)


    decoder_lstm = tf.keras.layers.LSTM(512,return_sequences=True, activation= 'tanh', name = 'justification_lstm')
    lstm_out = decoder_lstm(decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state])


    justification_output = tf.keras.layers.Dense(vocab_size, name = 'justification_output' )(lstm_out)

    model = tf.keras.Model(
        inputs = [head_idx,          # User ID
            tail_idx,          # Item ID
            brand_idx,         # Brand ID
            cat_idx,           # Category ID
            input_ids,         # BERT token IDs (Encoder)
            attn_mask,    # BERT padding mask (Encoder)
            decoder_inputs     # LSTM shifted targets (Decoder)
        ],
        outputs = [rating_output, justification_output],
        name="Explainable_Hybrid_Recommender",
    )

    return model

In [8]:
# def build_baseline_ablation_model(num_entities, vocab_size, max_seq_len=128, embed_dim=64):
#     # 1. Text Encoder Inputs
#     input_ids = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name='input_ids')
#     attn_mask = tf.keras.Input(shape=(max_seq_len,), dtype="int32", name='attention_mask')
#
#     # BERT Encoding
#     bert = TFBertModel.from_pretrained('bert-base-uncased')
#     bert.trainable = False
#     bert_output = bert(input_ids=input_ids, attention_mask=attn_mask)
#     bert_compressed = tf.keras.layers.Dense(64, activation='relu', name='bert_compression')(bert_output.pooler_output)
#
#     # 2. Collaborative Filtering Inputs (NO KNOWLEDGE GRAPH)
#     head_idx = tf.keras.Input(shape=(1,), dtype="int32", name='head_idx')
#     tail_idx = tf.keras.Input(shape=(1,), dtype="int32", name='tail_idx')
#
#     entity_emb_layer = tf.keras.layers.Embedding(input_dim=num_entities, output_dim=embed_dim, name='entity_emb')
#
#     h_emb = tf.keras.layers.Flatten()(entity_emb_layer(head_idx))
#     t_emb = tf.keras.layers.Flatten()(entity_emb_layer(tail_idx))
#
#     # 3. Fusion WITHOUT Brand or Category
#     merged = tf.keras.layers.concatenate([h_emb, t_emb, bert_compressed])
#     fusion_dense = tf.keras.layers.Dense(256, activation='relu', name='fusion_dense')(merged)
#     fusion_state = tf.keras.layers.Dropout(0.2)(fusion_dense)
#
#     # 4. Rating Head
#     rating_output = tf.keras.layers.Dense(1, activation='linear', name='rating_output')(fusion_state)
#
#     # 5. NLG Head
#     decoder_inputs = tf.keras.Input(shape=(None,), dtype="int32", name='decoder_inputs')
#     lstm_initial_state = tf.keras.layers.Dense(512, activation='tanh', name='lstm_initial_state')(fusion_state)
#     decoder_embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=256, name='decoder_embedding')(decoder_inputs)
#
#     decoder_lstm = tf.keras.layers.LSTM(512, return_sequences=True, activation='tanh', name='justification_lstm')
#     lstm_out = decoder_lstm(decoder_embedding, initial_state=[lstm_initial_state, lstm_initial_state])
#
#     justification_output = tf.keras.layers.Dense(vocab_size, name='justification_output')(lstm_out)
#
#     # Compile the Baseline
#     model = tf.keras.Model(
#         inputs=[head_idx, tail_idx, input_ids, attn_mask, decoder_inputs],
#         outputs=[rating_output, justification_output],
#         name="Baseline_No_KG_Recommender"
#     )
#
#     return model

# Initialize and Compile


In [9]:
vocab_size = tokenizer.vocab_size

# 1. Cast initial max IDs to strict integers
max_user_id = int(final_df['user_id_encoded'].max())
max_item_id = int(final_df['item_id_encoded'].max())

print('maxuserid ', max_user_id)
print('maxitemid ', max_item_id)
num_entities = max(max_user_id, max_item_id)

# 2. Handle Brands (Fill NaNs with 0 and cast to int)
brands_unique = final_df['brand'].dropna().unique()
brands_map = {brand : idx + num_entities + 1 for idx, brand in enumerate(brands_unique)}

final_df['brand_encoded'] = final_df['brand'].map(brands_map).fillna(0).astype(int)
print(final_df['brand_encoded'].head(3))

num_entities = max(num_entities, int(final_df['brand_encoded'].max()))

# 3. Handle Categories (Fix column name, fill NaNs, cast to int)
cats_unique = final_df['category'].dropna().unique()
cats_map = {category : idx + num_entities + 1 for idx, category in enumerate(cats_unique)}

final_df['category_encoded'] = final_df['category'].map(cats_map).fillna(0).astype(int)

# 4. Final strict integer cast for Keras
num_entities = int(max(num_entities, final_df['category_encoded'].max())) + 1
num_relations = 3

print(f"\nFinal num_entities passed to Keras: {num_entities} (Type: {type(num_entities)})")

# Build and compile the model
model = build_joint_explainability_model(num_entities, num_relations, vocab_size, 128, 64)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5, clipnorm=1.0),
    loss={
        'rating_output': 'mse',
        'justification_output': tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    },
    loss_weights={
        'rating_output': 1,
        'justification_output': 0.5
    },
    metrics={'rating_output': 'mae'},
)

maxuserid  15321
maxitemid  59428
0    59429
1    59430
2    59431
Name: brand_encoded, dtype: int64

Final num_entities passed to Keras: 77483 (Type: <class 'int'>)


I0000 00:00:1790153349.792502    7569 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification mode

Model: "Explainable_Hybrid_Recommender"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 head_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 tail_idx (InputLayer)       [(None, 1)]                  0         []                            
                                                                                                  
 brand_idx (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 cat_idx (InputLayer)        [(None, 1)]                  0         []                            
                                                                     

In [10]:
import numpy as np
from transformers import BertTokenizer


def create_tf_dataset(final_df, tokenizer, max_len = 128, batch_size = 16, is_training = True):
    encoder_tokenized = tokenizer(
        final_df['name'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='np',
    )


    print(f'Tokenizinf {len(final_df)}  Reviews')


    decoder_tokenized = tokenizer(
        final_df['reviews_text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='np',
    )


    input_ids = encoder_tokenized['input_ids']
    attn_mask = encoder_tokenized['attention_mask']


    decoder_inputs = decoder_tokenized['input_ids'][:, :-1]
    justification_targets = decoder_tokenized['input_ids'][:, 1:]

    inputs = {
        'input_ids': input_ids,
        'attention_mask': attn_mask,
        'decoder_inputs': decoder_inputs,
        "head_idx": final_df['user_id_encoded'].values.reshape(-1, 1),
        # "rel_idx": final_df['relation_encoded'].values.reshape(-1, 1),
        "tail_idx": final_df['item_id_encoded'].values.reshape(-1, 1),
        "brand_idx": final_df['brand_encoded'].values.reshape(-1, 1),
        "cat_idx": final_df['category_encoded'].values.reshape(-1, 1),

    }

    targets = {
        'rating_output':final_df['rating'].values.reshape(-1, 1),
        "justification_output": justification_targets
    }
    justification_weights = (justification_targets != 0).astype(np.float32)
    rating_weight = np.ones((len(final_df), 1), dtype = np.float32)
    sample_weights = {
        'rating_output': rating_weight,
        'justification_output': justification_weights
    }
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets, sample_weights))
    if is_training:
        dataset = dataset.shuffle(buffer_size = len(final_df), seed = 42)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
final_df_shuffled = final_df.sample(frac = 1, random_state = 42).reset_index(drop = True)
split_idx = int(len(final_df_shuffled) * 0.8)


train_idx = final_df_shuffled.iloc[:split_idx]
val_idx = final_df_shuffled.iloc[split_idx:]

train_dataset = create_tf_dataset(train_idx, tokenizer, max_len = 128, batch_size = 16, is_training = True)

val_dataset = create_tf_dataset(val_idx, tokenizer, max_len = 128, batch_size = 16, is_training = False)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights=True)],
)
#







/home/deven/miniconda3/envs/anchit/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Tokenizinf 80000  Reviews
Tokenizinf 20000  Reviews
Epoch 1/15


I0000 00:00:1790153417.662284    7660 cuda_dnn.cc:461] Loaded cuDNN version 92501
I0000 00:00:1790153418.244878    7654 service.cc:153] XLA service 0x7f0b5d3fd700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790153418.244914    7654 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.4.0; Runtime: 13.0.0; Toolkit: 13.0.0; DNN: 9.25.1)
I0000 00:00:1790153418.251101    7654 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790153418.313908    7661 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5000/5000 [==============================] - 835s 165ms/step - loss: 3.3094 - rating_output_loss: 1.8421 - justification_output_loss: 2.9346 - rating_output_mae: 1.0281 - val_loss: 3.0162 - val_rating_output_loss: 1.6132 - val_justification_output_loss: 2.8060 - val_rating_output_mae: 0.8902
Epoch 2/15
1178/5000 [======>.......................] - ETA: 8:32 - loss: 2.9843 - rating_output_loss: 1.5963 - justification_output_loss: 2.7761 - rating_output_mae: 0.9646 

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction



def evaluate_nlg_performance(model, val_dataset, tokenizer):
    print('Evaluating NLG')

    bleu_scores = []
    smoother = SmoothingFunction().method1
    for batch_inputs, batch_targets, batch_weights in val_dataset:
        predictions = model(batch_inputs, training = False)
        justification_logits = predictions[1]
        pred_token_ids = tf.argmax(justification_logits, axis = -1).numpy()
        true_token_ids = batch_targets['justification_output'].numpy()

        for i in range(0, len(true_token_ids)):
            true_text = tokenizer.decode(true_token_ids[i], skip_special_tokens=True).split()
            pred_text = tokenizer.decode(pred_token_ids[i], skip_special_tokens=True).split()
            if len(true_text) > 0:
               score = sentence_bleu([true_text], pred_text, smoothing_function=smoother)
               bleu_scores.append(score)
    final_score = np.mean(bleu_scores)
    print('Final Score = ', final_score)
    print("\n--- Example Generation ---")
    print(f"Target: {' '.join(true_text)}")
    print(f"Output: {' '.join(pred_text)}")

    return final_score
val_bleu = evaluate_nlg_performance(model, val_dataset, tokenizer)
print(f'BLEU Score = {val_bleu}')




In [ ]:
model.save_weights('actual_hybrid_weights.h5')

In [ ]:
def generate_justification(model, user_id, item_id, brand_id, cat_id, item_name, tokenizer, max_length=30, temperature=0.7, top_k=50):
    generated_sequence = [tokenizer.cls_token_id]

    head_tensor = tf.constant([[user_id]], dtype=tf.int32)
    tail_tensor = tf.constant([[item_id]], dtype=tf.int32)
    b_tensor = tf.constant([[brand_id]], dtype=tf.int32)
    c_tensor = tf.constant([[cat_id]], dtype=tf.int32)

    # Encode the actual product name to feed to BERT
    name_tokens = tokenizer(item_name, truncation=True, padding='max_length', max_length=128, return_tensors='tf')
    input_ids_tensor = name_tokens['input_ids']
    attn_mask_tensor = name_tokens['attention_mask']

    for _ in range(max_length):
        decoder_input = tf.constant([generated_sequence], dtype=tf.int32)

        predictions = model(
            [head_tensor, tail_tensor, b_tensor, c_tensor, input_ids_tensor, attn_mask_tensor, decoder_input],
            training=False
        )

        next_token_logits = predictions[1][0, -1, :] / temperature
        top_k_logits, top_k_indices = tf.math.top_k(next_token_logits, k=top_k)
        gumbel_probs = tf.nn.softmax(top_k_logits).numpy()
        chosen_idx = np.random.choice(len(top_k_indices), p=gumbel_probs)
        next_token_id = top_k_indices[chosen_idx].numpy()

        if next_token_id == tokenizer.sep_token_id:
            break

        generated_sequence.append(next_token_id)

    return tokenizer.decode(generated_sequence, skip_special_tokens=True)

In [ ]:
# Grab a sample from your validation split
sample = val_idx.iloc[0]

gen_text = generate_justification(
    baseline_model,
    user_id=sample['user_id_encoded'],
    item_id=sample['item_id_encoded'],
    brand_id=sample['brand_encoded'],
    cat_id=sample['category_encoded'],
    item_name=sample['name'],          # <-- Added this line!
    tokenizer=tokenizer,
    max_length=12,
    temperature=0.2,   # <-- Drop from 0.7 to 0.2
    top_k=5
)

print("🎯 TRUE REVIEW:")
print(sample['reviews_text'])
print("\n🤖 GENERATED JUSTIFICATION:")
print(gen_text)

In [ ]:
print("Starting Baseline Ablation Training (No KG Features)...")

baseline_history = baseline_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
)

In [ ]:
def generate_baseline_justification(model, user_id, item_id, item_name, tokenizer, max_length=30, temperature=0.7, top_k=50):
    generated_sequence = [tokenizer.cls_token_id]
    head_tensor = tf.constant([[user_id]], dtype=tf.int32)
    tail_tensor = tf.constant([[item_id]], dtype=tf.int32)

    name_tokens = tokenizer(item_name, truncation=True, padding='max_length', max_length=128, return_tensors='tf')
    input_ids_tensor = name_tokens['input_ids']
    attn_mask_tensor = name_tokens['attention_mask']

    for _ in range(max_length):
        decoder_input = tf.constant([generated_sequence], dtype=tf.int32)
        # Note: Brand and Category tensors removed to match Baseline architecture
        predictions = model(
            [head_tensor, tail_tensor, input_ids_tensor, attn_mask_tensor, decoder_input],
            training=False
        )

        next_token_logits = predictions[1][0, -1, :] / temperature
        top_k_logits, top_k_indices = tf.math.top_k(next_token_logits, k=top_k)
        gumbel_probs = tf.nn.softmax(top_k_logits).numpy()
        chosen_idx = np.random.choice(len(top_k_indices), p=gumbel_probs)
        next_token_id = top_k_indices[chosen_idx].numpy()

        if next_token_id == tokenizer.sep_token_id:
            break
        generated_sequence.append(next_token_id)

    return tokenizer.decode(generated_sequence, skip_special_tokens=True)

In [ ]:
# 1. Convert the split DataFrames into TF Datasets
print("Preparing evaluation datasets...")
cold_dataset = create_tf_dataset(final_df_cold, tokenizer, max_len=128, batch_size=16, is_training=False)
head_dataset = create_tf_dataset(final_df_head, tokenizer, max_len=128, batch_size=16, is_training=False)

# 2. Evaluate Rating Accuracy (MAE)
print("\n--- RATING PREDICTION EVALUATION ---")
print("Evaluating on HEAD items (5+ ratings)...")
head_metrics = model.evaluate(head_dataset)

print("\nEvaluating on COLD items (<5 ratings)...")
cold_metrics = model.evaluate(cold_dataset)

# 3. Evaluate Justification Quality (BLEU)
print("\n--- TEXT GENERATION (NLG) EVALUATION ---")
print("Calculating BLEU score for HEAD items...")
head_bleu = evaluate_nlg_performance(model, head_dataset, tokenizer)

print("\nCalculating BLEU score for COLD items...")
cold_bleu = evaluate_nlg_performance(model, cold_dataset, tokenizer)